<div align="center">
  <img src="https://raw.githubusercontent.com/erico-cristiam/ilpf-offline-agent/main/public/agroflora-cover.jpeg" width="720" alt="AGROFLORA IA">
</div>

# AGROFLORA IA — educação em ILPF com Gemma 3n

Protótipo de uma tutora de IA offline para pequenos e médios produtores da Amazônia, com foco em Integração Lavoura-Pecuária-Floresta (ILPF).

Build with Gemma: Amazon Eco-Hack — UFAC

English summary: AGROFLORA IA combines Gemma 3n E2B with a local retrieval layer to provide accessible, source-grounded ILPF education. The web prototype demonstrates the mobile experience; the intended product runs offline on Android devices.

## 1. Problema

A ILPF pode apoiar a recuperação de áreas produtivas, a diversificação de renda e o uso mais eficiente da terra. Entretanto, sua implantação exige diagnóstico, planejamento e acompanhamento técnico. Em comunidades rurais da Amazônia, a conectividade limitada também dificulta o acesso contínuo a materiais de capacitação.

Nossa pergunta foi: como oferecer orientação educacional acessível, verificável e útil mesmo onde a internet é instável ou inexistente?

## 2. Solução proposta

A AGROFLORA IA é uma tutora educacional específica para ILPF. Ela combina:

- Gemma 3n E2B para gerar explicações em português simples;
- RAG local para recuperar trechos técnicos, científicos e legais;
- indicação das fontes utilizadas em cada resposta;
- perfil básico da propriedade para contextualizar a explicação;
- trilhas curtas, biblioteca e exercícios de aprendizagem;
- arquitetura planejada para funcionar offline em celulares Android.

O protótipo não substitui assistência técnica, parecer jurídico ou decisão de órgão ambiental.

## 3. Arquitetura

```text
Pergunta do produtor
        ↓
Normalização e busca lexical local
        ↓
Seleção dos trechos mais relevantes
        ↓
Prompt com perfil + contexto + regras de segurança
        ↓
Gemma 3n E2B
        ↓
Resposta educacional + fontes recuperadas
```

Novos documentos entram na camada RAG. Portanto, atualizar a base não exige treinar novamente o Gemma. Um ajuste fino futuro seria considerado apenas para alterar comportamento, linguagem ou formato, não para armazenar fatos que mudam com o tempo.

## 4. Preparação do ambiente

No Kaggle, selecione um acelerador GPU em Notebook options. Para executar o modelo sem internet, adicione os pesos do Gemma 3n E2B Instruct em Add Input → Models. A célula de instalação pode ser ignorada se a versão disponível do Transformers já oferecer suporte ao Gemma 3n.

In [ ]:
# Execute somente se o import do Gemma 3n falhar na imagem atual do Kaggle.
# A instalação requer que a opção Internet esteja habilitada.
# %pip install -q -U "transformers>=4.53.0" accelerate sentencepiece

import json
import re
import unicodedata
from pathlib import Path
from typing import Any

import pandas as pd

try:
    import torch
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False

print({
    "torch_disponivel": TORCH_AVAILABLE,
    "gpu_disponivel": bool(TORCH_AVAILABLE and torch.cuda.is_available()),
})

## 5. Base de conhecimento

A base classifica cada fonte por autoridade. Leis oficiais têm prioridade em perguntas legais; publicações técnicas apoiam manejo e planejamento; páginas de divulgação são identificadas e usadas com ressalvas.

Para usar todos os trechos, envie `knowledge-base/chunks.json` como um Kaggle Dataset e anexe-o a este notebook. Se o arquivo não for encontrado, uma amostra incorporada mantém a demonstração executável.

In [ ]:
FALLBACK_KNOWLEDGE = [
    {
        "id": "decreto-6514-art-15",
        "title": "Decreto nº 6.514/2008 — embargo ambiental",
        "institution": "Presidência da República",
        "authority": "fonte_legal_primaria",
        "url": "https://www.planalto.gov.br/ccivil_03/_ato2007-2010/2008/decreto/d6514compilado.htm",
        "tags": ["embargo", "desembargo", "regularização"],
        "content": "A cessação da suspensão ou do embargo depende de decisão da autoridade ambiental depois da apresentação da documentação que regularize a obra ou atividade. A adoção de ILPF não produz desembargo automático.",
    },
    {
        "id": "lei-12651-regularizacao",
        "title": "Lei nº 12.651/2012 — CAR, PRA e recomposição",
        "institution": "Presidência da República",
        "authority": "fonte_legal_primaria",
        "url": "https://www.planalto.gov.br/ccivil_03/_ato2011-2014/2012/lei/l12651.htm",
        "tags": ["CAR", "PRA", "reserva legal", "recomposição"],
        "content": "A regularização ambiental rural envolve instrumentos como o Cadastro Ambiental Rural e os Programas de Regularização Ambiental. Projetos de recomposição devem observar os critérios legais e do órgão ambiental.",
    },
    {
        "id": "embrapa-ilpf-2021",
        "title": "ILPF: fundamentos e aplicações",
        "institution": "Embrapa",
        "authority": "referencia_tecnica",
        "url": "https://www.embrapa.br/tema-integracao-lavoura-pecuaria-floresta-ilpf",
        "tags": ["ILPF", "planejamento", "pastagem", "árvores"],
        "content": "A ILPF integra atividades agrícolas, pecuárias e florestais em rotação, consórcio ou sucessão. O planejamento começa pelo diagnóstico da propriedade, análise do solo, objetivos, recursos e escolha de componentes compatíveis.",
    },
    {
        "id": "embrapa-acre-2012",
        "title": "Experiências de implantação de ILPF no Acre",
        "institution": "Embrapa Acre",
        "authority": "referencia_tecnica_regional",
        "url": "https://www.embrapa.br/acre/busca-de-publicacoes",
        "tags": ["Acre", "implantação", "unidade demonstrativa"],
        "content": "Experiências locais ajudam a discutir implantação gradual, seleção dos componentes, acompanhamento técnico e avaliação de resultados produtivos e ambientais nas condições do Acre.",
    },
    {
        "id": "cultivando-futuro-solo",
        "title": "Cultivando o Futuro — qualidade do solo",
        "institution": "Embrapa Pecuária Sudeste",
        "authority": "referencia_tecnica",
        "url": "https://doi.org/10.4322/9786586819427",
        "tags": ["solo", "água", "nutrientes", "recuperação"],
        "content": "Sistemas integrados podem contribuir para recuperar funções físicas, químicas e biológicas do solo, desde que o manejo seja orientado por diagnóstico e acompanhamento.",
    },
    {
        "id": "cultivando-futuro-microclima",
        "title": "Cultivando o Futuro — microclima",
        "institution": "Embrapa Pecuária Sudeste",
        "authority": "referencia_tecnica",
        "url": "https://doi.org/10.4322/9786586819427",
        "tags": ["microclima", "sombra", "gado", "conforto térmico"],
        "content": "O componente arbóreo modifica radiação, vento, temperatura e umidade. Pode favorecer o conforto térmico animal, mas o sombreamento e a competição exigem monitoramento e ajuste do manejo.",
    },
    {
        "id": "amazonia-recuperacao",
        "title": "ILPF na Amazônia — recuperação de pastagens",
        "institution": "Embrapa Amazônia",
        "authority": "referencia_tecnica_regional",
        "url": "https://www.embrapa.br/amazonia-oriental",
        "tags": ["Amazônia", "pastagem degradada", "rotação"],
        "content": "Sistemas integrados podem recuperar áreas alteradas e renovar pastagens. A sequência adequada depende do clima, da logística, do custo e do perfil produtivo regional.",
    },
    {
        "id": "nobre-2020-amazon",
        "title": "Land-use and climate change risks in the Amazon",
        "institution": "Science Advances",
        "authority": "artigo_cientifico",
        "url": "https://doi.org/10.1126/sciadv.aba2949",
        "tags": ["Amazônia", "uso da terra", "clima"],
        "content": "A referência contextualiza riscos combinados de mudança do uso da terra e mudança climática na Amazônia e reforça a importância de conservar vegetação, solo e água.",
    },
]

def find_knowledge_file() -> Path | None:
    roots = [Path("/kaggle/input"), Path("../input"), Path(".")]
    for root in roots:
        if not root.exists():
            continue
        matches = [p for p in root.rglob("chunks.json") if "node_modules" not in p.parts]
        if matches:
            return matches[0]
    return None

knowledge_path = find_knowledge_file()
if knowledge_path:
    knowledge = json.loads(knowledge_path.read_text(encoding="utf-8"))
    knowledge_origin = str(knowledge_path)
else:
    knowledge = FALLBACK_KNOWLEDGE
    knowledge_origin = "amostra incorporada ao notebook"

print(f"Base carregada: {len(knowledge)} trechos — {knowledge_origin}")

In [ ]:
source_audit = (
    pd.DataFrame(knowledge)
    .groupby(["authority", "institution"], as_index=False)
    .size()
    .sort_values("size", ascending=False)
)
source_audit

## 6. Recuperação local de contexto

Para o protótipo de um dia, adotamos uma busca lexical transparente e leve. Termos encontrados no título recebem maior peso; tags recebem peso intermediário; o conteúdo recebe peso básico. Perguntas jurídicas também priorizam fontes legais primárias.

Em uma evolução do produto, essa etapa pode usar embeddings locais quantizados, mantendo a operação offline.

In [ ]:
STOP_WORDS = {
    "a", "ao", "aos", "as", "com", "como", "da", "das", "de",
    "do", "dos", "e", "em", "na", "nas", "no", "nos", "o",
    "os", "para", "por", "que", "um", "uma",
}

def terms(value: str) -> list[str]:
    normalized = unicodedata.normalize("NFD", value.lower())
    normalized = "".join(char for char in normalized if unicodedata.category(char) != "Mn")
    return [term for term in re.split(r"[^a-z0-9]+", normalized) if len(term) > 2 and term not in STOP_WORDS]

def retrieve(question: str, limit: int = 4) -> list[dict[str, Any]]:
    query_terms = terms(question)
    legal_question = bool(re.search(r"lei|legal|embargo|multa|app|reserva|car|pra", question, re.I))
    scored = []

    for chunk in knowledge:
        title_terms = terms(chunk["title"])
        tag_terms = terms(" ".join(chunk.get("tags", [])))
        content_terms = terms(chunk["content"])
        score = sum(
            (5 if term in title_terms else 0)
            + (3 if term in tag_terms else 0)
            + content_terms.count(term)
            for term in query_terms
        )
        if legal_question and chunk["authority"] == "fonte_legal_primaria":
            score += 20
        scored.append((score, chunk))

    return [chunk for _, chunk in sorted(scored, key=lambda item: item[0], reverse=True)[:limit]]

test_question = "Como recuperar uma pastagem degradada com ILPF?"
pd.DataFrame(retrieve(test_question))[["title", "institution", "authority"]]

## 7. Por que Gemma 3n E2B?

O Gemma 3n foi projetado para dispositivos cotidianos, incluindo celulares, tablets e notebooks. A variante E2B equilibra capacidade e eficiência e está alinhada ao destino móvel do projeto. O notebook usa a variante instruction-tuned para seguir instruções e responder em português.

Documentação oficial: https://ai.google.dev/gemma/docs/gemma-3n

## 8. Localização e carregamento do modelo

O código procura automaticamente uma pasta do Gemma 3n entre os Inputs do Kaggle. Se não encontrar, usa o identificador oficial do Hugging Face; essa segunda opção exige internet e a aceitação da licença Gemma. Qualquer falha ativa o modo RAG demonstrativo de forma transparente.

In [ ]:
MODEL_ID = "google/gemma-3n-E2B-it"
RUN_GEMMA = True  # Mude para False para testar apenas a recuperação.

def find_gemma_directory() -> str | None:
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return None
    for config_path in input_root.rglob("config.json"):
        normalized = str(config_path.parent).lower().replace("_", "-")
        if "gemma-3n" in normalized and ("e2b" in normalized or "e-2b" in normalized):
            return str(config_path.parent)
    return None

local_model_path = find_gemma_directory()
MODEL_SOURCE = local_model_path or MODEL_ID
print("Fonte do modelo:", MODEL_SOURCE)

In [ ]:
processor = None
model = None
MODEL_READY = False
MODEL_ERROR = None

if RUN_GEMMA:
    try:
        if not TORCH_AVAILABLE:
            raise RuntimeError("PyTorch não está disponível.")

        from transformers import AutoModelForImageTextToText, AutoProcessor

        dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else (
            torch.float16 if torch.cuda.is_available() else torch.float32
        )
        processor = AutoProcessor.from_pretrained(MODEL_SOURCE, local_files_only=bool(local_model_path))
        model = AutoModelForImageTextToText.from_pretrained(
            MODEL_SOURCE,
            torch_dtype=dtype,
            device_map="auto",
            low_cpu_mem_usage=True,
            local_files_only=bool(local_model_path),
        ).eval()
        MODEL_READY = True
        print("Gemma 3n E2B carregado.")
    except Exception as error:
        MODEL_ERROR = f"{type(error).__name__}: {error}"
        print("Gemma não foi carregado; o notebook continuará em modo RAG.")
        print(MODEL_ERROR)
else:
    print("Carregamento desativado por RUN_GEMMA=False.")

## 9. Prompt educacional e geração fundamentada

O prompt impede que o modelo trate a ILPF como solução automática para embargo, exige distinção entre níveis de autoridade e solicita validação profissional para decisões de manejo. Também pede texto simples, sem negrito, adequado à interface mobile.

In [ ]:
SYSTEM_PROMPT = """
Você é a AGROFLORA IA, tutora educacional sobre Integração Lavoura-Pecuária-Floresta na Amazônia.
Responda em português brasileiro simples, objetivo e acolhedor.
Use somente o contexto recuperado e não invente artigos, percentuais, resultados ou referências.
Diferencie fonte legal primária, referência técnica e conteúdo de divulgação.
Não use negrito, asteriscos duplos ou títulos em Markdown; use texto simples e listas com hífen.
Em perguntas legais, explique que a resposta é educacional, que ILPF não causa desembargo automático e que a decisão cabe à autoridade ambiental.
Em recomendações de manejo, solicite validação de assistência técnica.
Termine com Fontes consultadas e cite os números correspondentes ao contexto.
""".strip()

def build_context(chunks: list[dict[str, Any]]) -> str:
    blocks = []
    for index, chunk in enumerate(chunks, start=1):
        blocks.append(
            f"[{index}] {chunk['title']}\n"
            f"Instituição: {chunk['institution']}\n"
            f"Nível: {chunk['authority']}\n"
            f"Conteúdo: {chunk['content']}\n"
            f"URL: {chunk['url']}"
        )
    return "\n\n".join(blocks)

def retrieval_answer(chunks: list[dict[str, Any]]) -> str:
    excerpts = "\n\n".join(f"- {chunk['content']}" for chunk in chunks[:2])
    return (
        "O Gemma não está ativo nesta sessão. A busca local encontrou:\n\n"
        + excerpts
        + "\n\nConsulte as fontes recuperadas e valide decisões legais ou de manejo com profissionais e órgãos competentes."
    )

def answer_question(
    question: str,
    property_size: str = "até 50 hectares",
    activity: str = "pecuária",
) -> tuple[str, list[dict[str, Any]], str]:
    chunks = retrieve(question, limit=4)
    if not MODEL_READY:
        return retrieval_answer(chunks), chunks, "RAG demonstrativo"

    prompt = (
        f"{SYSTEM_PROMPT}\n\n"
        f"Perfil informado: {property_size}; atividade principal: {activity}.\n\n"
        f"CONTEXTO:\n{build_context(chunks)}\n\n"
        f"PERGUNTA:\n{question}"
    )
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = {key: value.to(model.device) if hasattr(value, "to") else value for key, value in inputs.items()}
    input_length = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=350,
            do_sample=True,
            temperature=0.2,
            top_p=0.9,
        )
    generated_ids = output_ids[:, input_length:]
    answer = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    answer = answer.replace("**", "").strip()
    return answer, chunks, "Gemma 3n E2B + RAG"

## 10. Demonstração

A pergunta abaixo testa recuperação de pastagem, um dos usos mais relevantes da ILPF para o público do projeto.

In [ ]:
question = "Como posso começar a recuperar uma pastagem degradada usando princípios de ILPF?"
answer, sources, mode = answer_question(question)

print("Modo:", mode)
print("\nPergunta:", question)
print("\nResposta:\n", answer)
print("\nFontes recuperadas:")
for index, source in enumerate(sources, start=1):
    print(f"{index}. {source['title']} — {source['institution']}")

## 11. Avaliação mínima

Em vez de avaliar apenas fluência, verificamos se os contextos recuperados cobrem conceitos esperados. A equipe também deve revisar manualmente fidelidade às fontes, clareza, segurança e utilidade.

In [ ]:
EVALUATION_QUESTIONS = [
    {
        "question": "Como recuperar uma pastagem degradada usando princípios de ILPF?",
        "expected_terms": ["solo", "planejamento", "diagnóstico"],
    },
    {
        "question": "Quais benefícios as árvores podem oferecer ao rebanho?",
        "expected_terms": ["conforto", "sombra", "microclima"],
    },
    {
        "question": "A implantação de ILPF encerra automaticamente um embargo?",
        "expected_terms": ["autoridade", "embargo", "regularização"],
    },
]

evaluation_rows = []
for item in EVALUATION_QUESTIONS:
    retrieved = retrieve(item["question"], limit=4)
    retrieved_text = " ".join(chunk["content"] for chunk in retrieved).lower()
    found = [term for term in item["expected_terms"] if term.lower() in retrieved_text]
    evaluation_rows.append({
        "pergunta": item["question"],
        "conceitos_encontrados": ", ".join(found),
        "cobertura": f"{len(found)}/{len(item['expected_terms'])}",
        "primeira_fonte": retrieved[0]["institution"],
    })

pd.DataFrame(evaluation_rows)

## 12. Interface interativa opcional no notebook

Esta célula cria uma interface simples dentro da sessão do Kaggle. Ela serve para apresentação, mas não substitui o Live Demo: quando a sessão do notebook terminar, a interface também será encerrada.

In [ ]:
LAUNCH_NOTEBOOK_DEMO = False  # Mude para True durante a apresentação.

if LAUNCH_NOTEBOOK_DEMO:
    try:
        import gradio as gr

        def gradio_answer(question, property_size, activity):
            answer, sources, mode = answer_question(question, property_size, activity)
            citations = "\n".join(f"- {source['title']} — {source['institution']}" for source in sources)
            return f"Modo: {mode}\n\n{answer}\n\nFontes recuperadas:\n{citations}"

        demo = gr.Interface(
            fn=gradio_answer,
            inputs=[
                gr.Textbox(label="Pergunta sobre ILPF", lines=3),
                gr.Dropdown(["até 50 hectares", "51 a 200 hectares", "mais de 200 hectares"], value="até 50 hectares", label="Tamanho"),
                gr.Dropdown(["pecuária", "lavoura", "sistema misto", "planejamento"], value="pecuária", label="Atividade"),
            ],
            outputs=gr.Textbox(label="Resposta educacional", lines=14),
            title="AGROFLORA IA",
            description="Tutora educacional de ILPF para a Amazônia.",
        )
        demo.launch(inline=True, share=False, prevent_thread_lock=True)
    except Exception as error:
        print("Não foi possível abrir a interface opcional:", error)
else:
    print("Interface opcional desativada.")

## 13. Caminho para o aplicativo offline

O site do repositório é uma simulação navegável do produto. Na versão final para Android:

1. o modelo será convertido ou distribuído em formato quantizado compatível com um runtime móvel;
2. os trechos da base e o índice serão armazenados no próprio aparelho;
3. perguntas, recuperação e geração ocorrerão sem enviar dados da propriedade para a nuvem;
4. atualizações verificadas da base poderão ser baixadas quando houver conexão;
5. aparelhos sem capacidade para geração ainda poderão usar trilhas, biblioteca e busca local.

O notebook valida o fluxo de IA; o protótipo web valida a experiência do usuário; a próxima etapa valida desempenho e usabilidade em aparelhos Android reais.

## 14. Impacto, limites e próximos passos

Impacto esperado:

- ampliar o acesso a capacitação em ILPF;
- apoiar decisões mais informadas sem substituir profissionais;
- valorizar referências da Amazônia e do Acre;
- reduzir dependência de conectividade contínua;
- tornar visível a fonte de cada orientação.

Limitações atuais:

- a base é curada, mas ainda pequena;
- parte dos materiais complementares precisa de revisão especializada;
- a busca lexical não compreende todos os sinônimos;
- o protótipo não foi validado em campo;
- desempenho e tamanho do modelo precisam ser medidos em diferentes celulares.

Próximos passos: revisão com especialistas da UFAC e Embrapa, testes com produtores, embeddings locais, avaliação sistemática de respostas e empacotamento Android.

## 15. Referências principais

1. BRASIL. Decreto nº 6.514/2008, texto compilado. https://www.planalto.gov.br/ccivil_03/_ato2007-2010/2008/decreto/d6514compilado.htm
2. BRASIL. Lei nº 12.651/2012. https://www.planalto.gov.br/ccivil_03/_ato2011-2014/2012/lei/l12651.htm
3. EMBRAPA. Integração Lavoura-Pecuária-Floresta. https://www.embrapa.br/tema-integracao-lavoura-pecuaria-floresta-ilpf
4. EMBRAPA PECUÁRIA SUDESTE. Integração Lavoura-Pecuária-Floresta: Cultivando o Futuro, 2026. https://doi.org/10.4322/9786586819427
5. OLIVEIRA, T. K. de et al. Experiências com implantação de unidades de ILPF no Acre. Embrapa Acre, 2012.
6. NOBRE, C. A. et al. Land-use and climate change risks in the Amazon. Science Advances, 2020. https://doi.org/10.1126/sciadv.aba2949
7. PITMAN, A. J. et al. The impact of land cover change on climate, 2009. https://doi.org/10.1029/2007RG000242
8. BONAN, G. B. Ecological Climatology, 2016.
9. STULL, R. B. An Introduction to Boundary Layer Meteorology, 1988.
10. GOOGLE. Gemma 3n model overview. https://ai.google.dev/gemma/docs/gemma-3n

O catálogo completo e os critérios de curadoria estão no repositório do projeto.